# 00-03 · ETF Raw EDA Research Notebook

**대상:** ETF slot #03  
**단계:** `00_raw_eda`  
**목적:** 최종 전처리·정규화·라벨링을 하기 **전**, 원자료의 구조와 가설을 탐색한다.

**원칙:** `Observation → Implication → Decision`  
**시각화:** 별도 노트북으로 분리하지 않고 해당 분석 셀 바로 아래에 렌더링한다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

SLOT = 3
cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd] + list(cwd.parents) if (p/"config"/"etf_universe.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("config/etf_universe.csv가 있는 프로젝트 루트에서 실행하세요.")

sys.path.insert(0, str(ROOT))
from src.style import setup_notebook, banner, kpis
from src.eda_utils import read_universe, load_raw, engineer_basic, quality_summary, save_profile

setup_notebook()
UNIVERSE = read_universe(ROOT)
META = UNIVERSE.loc[UNIVERSE["slot"] == SLOT].iloc[0]
TICKER = str(META["ticker"])
NAME = str(META["name"])
RAW_PATH = ROOT / str(META["raw_file"])

banner("ETF #{:02d} · {} · {}".format(SLOT, TICKER, NAME),
       "Raw EDA only — no final preprocessing/model decision")
display(META.to_frame("value"))

## 01. 연구 질문·가설 계약

### Core RQ-A · 분포/정규화
- **H0:** 절대수익률 기준만으로도 극단 움직임을 안정적으로 표현할 수 있다.
- **H1:** rolling volatility/percentile 기준으로 표준화해야 안정적이다.

### Core RQ-B · 거래활동 확인
- **H0:** 극단수익률일과 일반일 사이 거래량·거래대금 차이가 없다.
- **H1:** 극단수익률일에 비정상 거래활동이 동반된다.

### Core RQ-C · 시작점 안정성
- **H0:** threshold/window를 조금만 바꾸면 candidate onset이 크게 달라진다.
- **H1:** 합리적인 범위에서는 비슷한 episode가 반복된다.

이 ETF에는 여기에 **Focused RQ 1개**를 추가한다.

In [ ]:
HYP = pd.read_csv(ROOT/"config"/"hypothesis_catalog.csv")
display(HYP[["key","short_name","rq","use_for"]])

primary = str(META.get("primary_hypothesis", "AUTO"))
if primary != "AUTO" and primary in set(HYP["key"]):
    FOCUS = HYP.loc[HYP["key"] == primary].iloc[0]
    banner("Focused Hypothesis · {} {}".format(FOCUS["key"], FOCUS["short_name"]), FOCUS["rq"])
    display(FOCUS.to_frame("value"))
else:
    FOCUS = None
    print("primary_hypothesis=AUTO — ETF taxonomy를 채운 뒤 H01~H10 중 하나를 지정하세요.")

## 02. 원자료 로드 · provenance
데이터를 아직 고치지 않는다. 행 수·컬럼·기간·단위를 먼저 기록한다.

In [ ]:
if not RAW_PATH.exists():
    print("[STOP] raw file 없음:", RAW_PATH)
    df_raw = pd.DataFrame()
else:
    df_raw = load_raw(RAW_PATH)
    kpis([
        ("rows", f"{len(df_raw):,}"),
        ("columns", len(df_raw.columns)),
        ("start", str(df_raw["date"].min().date()) if "date" in df_raw else "-"),
        ("end", str(df_raw["date"].max().date()) if "date" in df_raw else "-"),
    ])
    display(df_raw.head())
    display(df_raw.tail())

## 03. Schema · 결측 · 중복 · 시간축 무결성
- 필수 OHLCV 존재 여부
- 날짜 중복/역순
- 결측 집중 구간
- ETF별 원자료 단위 차이

In [ ]:
if not df_raw.empty:
    display(quality_summary(df_raw))
    print("duplicate dates:", int(df_raw["date"].duplicated().sum()) if "date" in df_raw else "N/A")
    print("date monotonic:", bool(df_raw["date"].is_monotonic_increasing) if "date" in df_raw else "N/A")

## 04. Raw OHLCV 구조
가격 scale, gap, OHLC 논리오류, 거래량 scale 변화를 먼저 본다.

In [ ]:
if not df_raw.empty:
    df = engineer_basic(df_raw)

    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(df["date"], df["close"])
    ax.set_title(f"{TICKER} Close Price")
    plt.show()

    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(df["date"], df["volume"])
    ax.set_title(f"{TICKER} Raw Volume")
    plt.show()

    if set(["open","high","low","close"]).issubset(df.columns):
        bad = (
            (df["high"] < df[["open","close"]].max(axis=1)) |
            (df["low"] > df[["open","close"]].min(axis=1)) |
            (df["high"] < df["low"])
        )
        print("OHLC logical violations:", int(bad.sum()))

## 05. 수익률 분포 · tail · robust scaling 필요성

시각화:
- return time series
- histogram
- Q-Q plot
- 연도별 boxplot

목적은 정규성을 '증명'하는 것이 아니라 **정규화 방식을 결정할 근거**를 만드는 것이다.

In [ ]:
if not df_raw.empty:
    r = df["ret_1d"].dropna()
    kpis([
        ("mean", f"{r.mean():.4%}"),
        ("std", f"{r.std():.4%}"),
        ("skew", f"{stats.skew(r):.3f}"),
        ("excess kurtosis", f"{stats.kurtosis(r, fisher=True):.3f}"),
        ("|ret| 95%", f"{r.abs().quantile(.95):.3%}"),
        ("|ret| 99%", f"{r.abs().quantile(.99):.3%}"),
    ])

    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(df["date"], df["ret_1d"])
    ax.axhline(0, linewidth=1)
    ax.set_title("Daily Return")
    plt.show()

    fig, ax = plt.subplots(figsize=(8,4))
    ax.hist(r, bins=60)
    ax.set_title("Return Distribution")
    plt.show()

    fig, ax = plt.subplots(figsize=(6,6))
    stats.probplot(r, dist="norm", plot=ax)
    ax.set_title("Q-Q Plot")
    plt.show()

    y = df.assign(year=df["date"].dt.year).dropna(subset=["ret_1d"])
    groups = [g["ret_1d"].values for _, g in y.groupby("year")]
    labels = [str(k) for k,_ in y.groupby("year")]
    if len(groups) >= 2:
        fig, ax = plt.subplots(figsize=(14,5))
        ax.boxplot(groups, labels=labels, showfliers=False)
        ax.set_title("Return by Year")
        plt.xticks(rotation=45)
        plt.show()

## 06. Volatility 구조
- rolling volatility 20/60
- squared return autocorrelation
- 평시 변동성이 시간에 따라 안정적인지 확인

In [ ]:
if not df_raw.empty:
    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(df["date"], df["roll_vol_20"], label="20D")
    ax.plot(df["date"], df["roll_vol_60"], label="60D")
    ax.set_title("Rolling Volatility")
    ax.legend()
    plt.show()

    sq = df["ret_1d"].pow(2).dropna()
    if len(sq) > 100:
        display(acorr_ljungbox(sq, lags=[5,10,20], return_df=True))

## 07. 거래활동과 가격 변화 · Core RQ-B

raw volume 자체보다 rolling median 대비 ratio, robust z-score를 우선 본다.

In [ ]:
if not df_raw.empty:
    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(df["date"], df["volume_ratio_20"])
    ax.axhline(1, linewidth=1)
    ax.set_title("Volume / 20D Rolling Median")
    plt.show()

    x = df[["abs_ret_1d","volume_robust_z_20"]].dropna()
    if len(x) > 30:
        rho, p = stats.spearmanr(x["abs_ret_1d"], x["volume_robust_z_20"])
        print(f"Spearman rho={rho:.4f}, p={p:.4g}")

        fig, ax = plt.subplots(figsize=(7,5))
        ax.hexbin(x["volume_robust_z_20"], x["abs_ret_1d"], gridsize=35, mincnt=1)
        ax.set_xlabel("Volume robust z (20D)")
        ax.set_ylabel("|Return|")
        ax.set_title("Trading Activity vs Absolute Return")
        plt.show()

        q95 = df["abs_ret_1d"].quantile(.95)
        extreme = df.loc[df["abs_ret_1d"] >= q95, "volume_robust_z_20"].dropna()
        normal = df.loc[df["abs_ret_1d"] < q95, "volume_robust_z_20"].dropna()
        if len(extreme) > 5 and len(normal) > 20:
            stat, p = stats.mannwhitneyu(extreme, normal, alternative="two-sided")
            print(f"Mann–Whitney U p={p:.4g}")

## 08. Candidate Event Map · Core RQ-C

아직 label을 확정하지 않는다.

서로 다른 기준으로 후보만 생성한다.
- abs-return 95 percentile
- return / rolling volatility = 2.0 / 2.5 / 3.0
- 필요하면 거래량 동반 여부 추가

**목표:** threshold 민감도를 먼저 본다.

In [ ]:
if not df_raw.empty:
    df["ret_z20"] = df["ret_1d"] / df["roll_vol_20"].replace(0, np.nan)
    cand = pd.DataFrame({
        "date": df["date"],
        "ret_1d": df["ret_1d"],
        "ret_z20": df["ret_z20"],
        "volume_ratio_20": df["volume_ratio_20"],
        "abs_q95": df["abs_ret_1d"] >= df["abs_ret_1d"].quantile(.95),
        "z2": df["ret_z20"] >= 2.0,
        "z25": df["ret_z20"] >= 2.5,
        "z3": df["ret_z20"] >= 3.0,
    })
    display(cand.loc[cand[["abs_q95","z2","z25","z3"]].any(axis=1)].tail(40))

    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(df["date"], df["ret_z20"])
    for level in [2.0, 2.5, 3.0]:
        ax.axhline(level, linewidth=1)
    ax.set_title("Return / 20D Rolling Volatility")
    plt.show()

## 09. 큰 상승 이후 경로 · 지속 vs 반전

후행 outcome은 **EDA용**으로만 본다.
실제 detector input에는 미래값을 넣지 않는다.

In [ ]:
if not df_raw.empty:
    for h in [1,3,5,10]:
        df[f"fwd_ret_{h}d"] = df["close"].shift(-h) / df["close"] - 1

    q95p = df["ret_1d"].quantile(.95)
    events = df.loc[
        df["ret_1d"] >= q95p,
        ["date","ret_1d","volume_ratio_20"] + [f"fwd_ret_{h}d" for h in [1,3,5,10]]
    ]
    display(events.tail(30))

## 10. Benchmark / Peer 비교
metadata가 있을 때만 실행한다.

질문: 큰 움직임이 broad market과 같은 움직임인가, ETF 고유 움직임인가?

In [ ]:
benchmark = str(META.get("benchmark_ticker","")).strip()
if benchmark and benchmark.lower() != "nan":
    peer_meta = UNIVERSE.loc[UNIVERSE["ticker"].astype(str) == benchmark]
    if len(peer_meta):
        bpath = ROOT / str(peer_meta.iloc[0]["raw_file"])
        if bpath.exists():
            b = engineer_basic(load_raw(bpath))[["date","ret_1d"]].rename(columns={"ret_1d":"bench_ret"})
            z = df.merge(b, on="date", how="inner")
            z["abnormal_ret"] = z["ret_1d"] - z["bench_ret"]
            z["rolling_corr_60"] = z["ret_1d"].rolling(60).corr(z["bench_ret"])

            fig, ax = plt.subplots(figsize=(14,4))
            ax.plot(z["date"], z["rolling_corr_60"])
            ax.set_title(f"60D Rolling Correlation vs {benchmark}")
            plt.show()

            fig, ax = plt.subplots(figsize=(14,4))
            ax.plot(z["date"], z["abnormal_ret"])
            ax.axhline(0, linewidth=1)
            ax.set_title("Benchmark-adjusted Return")
            plt.show()
        else:
            print("benchmark raw file 없음:", bpath)
    else:
        print("benchmark ticker가 universe에 없음:", benchmark)
else:
    print("benchmark_ticker 미지정 — config에서 지정 후 실행")

## 11. Optional Multivariate EDA · PCA

PCA는 예측모델이 아니라, raw feature가 실제로 소수 잠재축으로 압축되는지 보는 진단이다.
설명분산이 낮으면 그대로 기각한다.

In [ ]:
if not df_raw.empty:
    feats = [c for c in [
        "ret_1d","abs_ret_1d","roll_vol_20","volume_robust_z_20","drawdown","range_pct"
    ] if c in df.columns]
    z = df[["date"] + feats].dropna().copy()

    if len(z) >= 100 and len(feats) >= 4:
        X = RobustScaler().fit_transform(z[feats])
        pca = PCA(n_components=min(4, len(feats)))
        pcs = pca.fit_transform(X)
        display(pd.DataFrame({
            "PC":[f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
            "explained_ratio":pca.explained_variance_ratio_,
            "cumulative":np.cumsum(pca.explained_variance_ratio_)
        }))
        fig, ax = plt.subplots(figsize=(7,6))
        ax.scatter(pcs[:,0], pcs[:,1], s=10)
        ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
        ax.set_title("PCA State Map")
        plt.show()
    else:
        print("PCA skip: 충분한 행/feature 없음")

## 12. 연구 기록 · Observation → Implication → Decision

### Observation
- 무엇이 보였는가?

### Implication
- 전처리/정규화/threshold/label에 어떤 의미인가?

### Decision
- 다음 단계에서 무엇을 채택·비교·보류할 것인가?

---

### Core RQ-A
- H0 유지 / H1 지지 / 판단보류:
- 근거:

### Core RQ-B
- H0 유지 / H1 지지 / 판단보류:
- 근거:

### Core RQ-C
- H0 유지 / H1 지지 / 판단보류:
- 근거:

### Focused RQ
- 가설:
- 근거:
- 다음 실험:

In [ ]:
if not df_raw.empty:
    profile = {
        "slot": int(SLOT),
        "ticker": TICKER,
        "name": NAME,
        "rows": int(len(df)),
        "start": str(df["date"].min().date()),
        "end": str(df["date"].max().date()),
        "ret_mean": float(df["ret_1d"].mean()),
        "ret_std": float(df["ret_1d"].std()),
        "abs_ret_q95": float(df["abs_ret_1d"].quantile(.95)),
        "abs_ret_q99": float(df["abs_ret_1d"].quantile(.99)),
        "median_volume_ratio_20": float(df["volume_ratio_20"].median(skipna=True)),
        "primary_hypothesis": str(META.get("primary_hypothesis","AUTO")),
    }
    out = ROOT/"data"/"interim"/"00_profiles"/f"{TICKER}.json"
    save_profile(profile, out)
    print("EDA profile saved:", out)
    display(pd.Series(profile, name="value").to_frame())

## 13. 종료 조건

이 노트북은 **최종 threshold / 최종 정규화 / 최종 label / 최종 모델을 정하지 않는다.**

대신 다음을 남기면 성공이다.
1. raw 데이터 품질과 scale
2. 수익률·변동성·거래활동 분포
3. candidate event 구조
4. threshold/window 민감도
5. benchmark/peer 비교 필요성
6. 다음 전처리 단계의 명시적 결정사항